# **Procesamiento de Lenguaje Natural**: Preprocesado de documentos con NLTK y Spacy


**Pablo Martínez Olmos, Vanessa Gómez Verdejo, Emilio Parrado Hernández, Angel Navia Vázquez**

  * v1.1 (January 2025) Revised and updated version
  
Departamento de Teoría de la Señal y Comunicaciones

**Universidad Carlos III de Madrid**

<img src='http://www.tsc.uc3m.es/~navia/figures/logo_uc3m_foot.jpg' width=400 />


In [1]:
# Ejecutamos este código para preparar el contexto.
%matplotlib inline
# Figures plotted inside the notebook
%config InlineBackend.figure_format = 'svg'
# High quality figures
import matplotlib.pyplot as plt

In [2]:
import sys
!{sys.executable} -m pip install numpy nltk spacy pandas



  Using cached nltk-3.9.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-0.4.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached typer_slim-0.23.0-py3-none-any.whl.metadata (4.2 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached confection-0.1.5-py3-none-any.whl.metadata (19 kB)
  Using cached typer-0.23.0-py3-none-any.whl.metadata (16 kB)
  Using cached cloudpathlib-0.23.0-py3-none-any.whl.metadata (16 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.m

In [3]:
!/opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.5 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3


In [1]:
# Estas librerías pueden dar problemas de dependencias, conviene instalarlas conjuntamente, para que pip resuelva correctamente las dependencias
!pip install --upgrade numpy nltk spacy pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:

!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install -U numpy pandas nltk spacy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 11.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 15.0 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: pandas━━━━━━━━━━━━━━━━━━ 0/2 [numpy]
    Found existing installation: pandas 2.2.1 0/2 [numpy]
    Uninstalling pandas-2.2.1:0m╺━━━━━━━━━━━━━━━━━━━ 1/2 [pandas]
      Successfully uninstalled pandas-2.2.1━━━━━━━━━━━━━━━━━━━ 1/2 [pandas]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
contourpy 1.2.0 requires numpy<2.0,>=1.20, but you have numpy 2.4.2 which is incompatible.
matplotlib 3.8.2 requires numpy<2,>=1.21, but you have numpy 2.4.2 which is inc

In [7]:
import numpy as np
import nltk
import spacy
import pandas as pd

print(f"NumPy version: {np.__version__}")
print(f"NLTK version: {nltk.__version__}")
print(f"Spacy version: {spacy.__version__}")
print(f"Pandas version: {pd.__version__}")

# 2025:
# NumPy version: 2.2.4
# NLTK version: 3.9.1
# Spacy version: 3.8.4
# Pandas version: 2.2.3

# 2026:
# NumPy version: 2.4.2
# NLTK version: 3.9.2
# Spacy version: 3.8.11
# Pandas version: 3.0.0

NumPy version: 1.26.4
NLTK version: 3.9.2
Spacy version: 3.8.11
Pandas version: 2.2.1


# 1 - Introducción

En esta sesión vamos a ver cómo trabajar con nuestros datos cuando éstos son **cadenas de texto**. A diferencia de los **datos categóricos**, en los que tenemos cadenas de texto asociadas a diferentes categorías y que podemos codificar fácilmente (por ejemplo, con un **one-hot-encoding**), cuando hablamos aquí de información textual nos referimos a frases, documentos y/o corpus de datos con estructura mucho más compleja. Idealmente, a partir de estos datos textuales tenemos que extraer la información necesaria (a poder ser incluyendo contenido semántico) y **vectorizarla adecuadamente para poder utilizar o usar modelos de aprendizaje automático** a partir de ella.


**Ejemplo de datos categóricos:**

`Profesión: "ingeniero"`

`Residencia: "Valladolid"`

`Día de la semana: "martes"`

Codificación "one-hot" (se activa una entre N posibilidades):

`"ingeniero" ->   0,0,0,0,0,0,1,0,0,0,0,0`

`"abogado"     ->   0,0,1,0,0,0,0,0,0,0,0,0`

`(...)`

**Ejemplo de datos textuales:**

`After visiting the Capitoline Museum, the Piazza Campidoglio is a great place to soak up classical Rome "al fresco." Also, there's lots for kids to (safely) climb on here. Just across the Tiber from Piazza Navona, Castel Sant Angelo is a museum full of fascinating treasures. Also, the castle's old moat is now a car-free park with playgrounds and space to run free! This is a mostly car-free area where kids can run and chase pigeons in the shadow of mythical fountains. Piazza Navona is especially wonderful in winter when it becomes a Christmas market.`

En general, gran parte de la forma en que nos comunicamos hoy en día es a través de **texto escrito**, ya sea en servicios de mensajería, medios sociales y/o correo electrónico. Nos encontramos con tareas de **creciente granularidad o complejidad**:

- **Valoración de opiniones** en servicios/aplicaciones como TripAdvisor, Booking, Amazon, etc.

- **Filtrado por contenido**: ejemplos clásicos son la categorización de noticias, o la identificación de correos "spam".

- **Recuperación de información** en grandes volúmenes de datos textuales.

- **Asistentes tipo chatbot**. Guiado de tareas sencillas usando lenguaje natural.

- **Traducción automática**. Obtención automática de traducciones de textos a una gran variedad de idiomas, incluso con posibilidad de tiempo real.

- **Modelos de Lenguaje (LLMs, IAs)**, han representado la última revolución en aplicaciones del procesado de lenguaje natural. Los límites y alcance de dichos modelos están todavía por ver.

Además, gracias a los **conversores voz-texto-voz**, cualquier mensaje vocal también puede procesarse como texto (multimodalidad).




El tratamiento automatizado de este tipo de información se denomina **procesamiento del lenguaje natural** (**Natural Language Processing, NLP**).
El NLP es un **subcampo de la lingüística, la informática y la inteligencia artificial** que se ocupa de las interacciones entre los ordenadores (o procesadores) y el lenguaje humano; en particular engloba un conjunto de técnicas para permitir que los ordenadores procesen y analicen grandes cantidades de texto.

# 2 - Canalización ("Pipeline") para el procesado de texto

Como sabemos, los algoritmos de **Aprendizaje Automático (ing. Machine Learning, ML**) procesan números, no palabras, por lo que necesitamos **transformar el texto en números significativos que contengan la información relevante de los documentos**. Este proceso de conversión de texto a números es lo que llamaremos **vectorización**.

No obstante, para tener una representación útil, se requieren normalmente algunos pasos previos de **preprocesamiento** que limpien y homogenizen los documentos: **tokenización, eliminación de *stop-words*, lematización**, etc.
La siguiente figura muestra los diferentes pasos que debemos seguir para procesar nuestros documentos hasta poder ser utilizados por nuestro modelo de aprendizaje:

<img src="http://www.tsc.uc3m.es/~navia/figures/PipelineNLP.png" width="80%">



A lo largo de este notebook, veremos las **herramientas** que tenemos disponibles en **Python** para llevar a cabo todos estos pasos. Concretamente, nos centraremos en el uso de las siguientes librerias:

* [**NLTK, Natural Language ToolKit**](https://www.nltk.org/). Esta libreria es una excelente biblioteca de NLP escrita en Python por expertos tanto del mundo académico como de la industria e incluye una **amplia variedad de utilidades**. NLTK permite crear aplicaciones con datos textuales rápidamente, ya que proporciona un conjunto de **clases básicas para trabajar con corpus de datos**, incluyendo colecciones de textos (corpus), listas de palabras clave, clases para representar y operar con datos de tipo texto (documentos, frases, palabras) así como otras funciones para tareas comunes de NLP.

* [**SpaCy**](https://spacy.io/) es una biblioteca de código abierto para procesamiento avanzado del lenguaje natural en Python. **SpaCy está diseñado específicamente para su uso en producción**. A diferencia de NLTK, SpaCy tiene una **estructura orientada a objetos**. Por ejemplo, al tokenizar texto, cada token es un objeto con atributos y propiedades específicas. SpaCy admite más de **64 idiomas**, incluyendo modelos estadísticos ya entrenados para [17 de ellos](https://spacy.io/usage/models) (incluyendo modelos avanzados tipo [**transformers**](https://spacy.io/usage/v3)).


* [**Gensim**](https://pypi.org/project/gensim/) es otra librería de Python para el **modelado por temáticas (*topic modeling*), la indexación de documentos y tareas de recuperación de la información para documentos**. Está diseñada para operar con grandes cantidades de información (con **implementaciones eficientes y paralelizables/distribuidas**) y nos va a ser de gran ayuda para la vectorización de nuestros corpus de datos una vez preprocesados.

**NOTA:** Utilizaremos indistintamente estas librerías para diferentes tareas. Es **MUY IMPORTANTE** ser consciente de qué tipo de dato de entrada y salida se maneja en cada caso, pues puede ser necesario **transformar la información** al migrar de una herramienta a otra.   

# 3 - Accediendo a un corpus: las bases de datos de NLTK.

**NLTK incluye diferentes corpus de datos** para probar nuestras herramientas. Podemos encontrar información sobre todos ellos en [corpus NLTK](https://www.nltk.org/book/ch02.html).

A pesar de que también podemos usar NLTK para realizar el preprocesamiento de texto, usaremos [spaCy](https://spacy.io/) para tal fin.


## 3.1 - El objeto CorpusReader
Para empezar a trabajar vamos a utilizar el corpus **movie_reviews**, uno de los corpus de datos incluidos en NLTK y formado por 2000 documentos de texto con reseñas de diferentes películas donde además se indica si estas reseñas son positivas o negativas.

La siguiente celda de código muestra cómo cargar el corpus:


In [ ]:
from nltk.corpus import movie_reviews
nltk.download('movie_reviews')

movie_reviews

Al cargar el corpus, se genera un **objeto de tipo `CorpusReader`**, denominado `movie_reviews`, con el contenido del corpus. Podemos ver todos los atributos y métodos asociados a la clase usando el comando `dir()` (no usaremos todos los métodos/atributos, veremos a continuación los más importantes). La documentación completa del paquete se puede consultar en el link de [NLTK corpus reader](https://www.nltk.org/api/nltk.corpus.reader.html).  


In [ ]:
dir(movie_reviews)

Un método importante en `CorpusReader` es **`fileids`**, que nos proporciona una lista indicando qué documentos componen este corpus. Veamos **cuántos documentos componen el corpus y los nombres de los 10 primeros documentos**:

In [ ]:
print("Este corpus contiene {} documentos.".format(len(movie_reviews.fileids())))
doc_ids = movie_reviews.fileids()
print("Los 10 primeros nombres de documentos son:")
for k in range(10):
  print(doc_ids[k])

También podemos ver las categorías de este corpus con el método **`categories`**. En este caso vemos que el corpus está etiquetado para indicar si un documento se refiere a un **comentario positivo o negativo**:

In [ ]:
movie_reviews.categories()

Vemos pues que hay dos categorías: positiva y negativa,
 según el tipo de comentario que ha recibido la película. Podemos acceder a la **categoría de cada uno de los textos** en el corpus:

In [ ]:
movie_reviews.categories('neg/cv000_29416.txt')

Observamos que el propio nombre del documento nos indica la categoría, pero esto no tiene porque ser siempre así, es necesario confiar en el método "`categories`".

Si queremos, podemos **acceder a un documento específico** en el corpus y extraer su contenido original con el método **`raw`**.

In [ ]:
raw_text = movie_reviews.raw('neg/cv000_29416.txt')
print(raw_text)

Podemos comprobar el typo de la variable `raw_text` y vemos que es un **string** (con todas los métodos asociados a los tipos *string*). Observamos que los textos ya en origen han sido sometidos a cierto preprocesado, por ejemplo, ya no hay mayúsculas...

Podemos confirmar el tipo y los primeros 40 caracteres de este documento como:

In [ ]:
print(type(raw_text))

print('\n The total number of characters in the document is %d, and these are the first 100 characters:\n' %(len(raw_text)))

print(raw_text[:100])



Usando los métodos asociados al tipo string podríamos hacer un **preprocesado básico de los textos**, por ejemplo **separando el texto en frases** usando el punto como separador:

In [ ]:
frases = raw_text.split('.')
print('Primera frase:')
print(frases[0])
print('\nSegunda frase:')
print(frases[1])


O podemos **separar en palabras** la segunda frase usando el espacio como separador:

In [ ]:
print(frases[1].split())

No obstante, las **posibilidades de procesado usando strings de python son muy limitadas** y la complejidad de los lenguajes es grande (puede haber otros separadores de frases o palabras, por ejemplo), por eso recurrimos a los **método específicos y especializados** incluidos en las librerías anteriormente mencionadas.

# 4 - Preprocesado del corpus

Antes de transformar los datos de entrada de texto en una representación vectorial, necesitamos **estructurar y limpiar el texto**, conservando en la medida de lo posible la información que permita capturar el contenido semántico del corpus.

Para ello, el **procesado típico de NLP** aplica los siguientes pasos:

1. Tokenización
2. Homogeneización
3. Limpieza

## 4.1 - Tokenization

**Tokenización** es el **proceso de dividir el texto dado en elementos individuales llamados tokens**. Las palabras, los números, los signos de puntuación y otros pueden ser considerados como tokens.

Ya hemos visto que usando strings de python podríamos dividir el corpus en frases o palabras, pero **NLTK incluye mejores funciones genéricas** para hacer estas operaciones sobre cualquier cadena de texto. En concreto, tiene dos funciones:

- **`sent_tokenize`: es un tokenizador de frases**. Este tokenizador divide un texto en una lista de oraciones. Para decidir dónde empieza o acaba una frase NLTK tiene un **modelo pre-entrenado para el idioma específico en el que estemos trabajando**. Este modelo se carga mediante `nltk.download('punkt_tab')`.

- **`word_tokenize`/`wordpunct_tokenize`:  Divide un texto en tokens** (palabras, siglas, abreviaturas u otros caracteres individuales, tales como números, signos de puntuación, emojis, etc.).

In [ ]:
# Ejemplo de tokenización usando NLTK:
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

# Extraemos un primer texto del corpus
raw_text = movie_reviews.raw(doc_ids[0])

# Dividimos el texto en frases:
frases = sent_tokenize(raw_text)

In [ ]:
print("sent_tokenize genera una lista de strings:")
print(type(frases))
print(type(frases[0]))

# Mostramos las 5 primeras frases
print("\nEstas son las 5 primeras frases:")
for k in range(5):
  print(frases[k])

Observamos que **conserva los separadores de frases** (".", "?").

In [ ]:
# Podemos dividir la primera frase en palabras:
print("word_tokenize toma un string y lo convierte en lista de strings (tokens):")
from nltk.tokenize import word_tokenize
tokens = word_tokenize(frases[0])
print(tokens)

Onservamos que **los signos de puntuación se convierten también en "tokens"**.

### Ejercicio:

Utilizando una "list comprehension" procese todas las frases del documento número 8 y obtenga los tokens que tengan más de un caracter. Mostrar los tokens de las frases número 3, 4 y 7 antes y después del filtrado de tokens. Identifique algunos de los tokens eliminados.

In [ ]:
# Procesamos el texto número 8
raw_text = <INSERT CODE HERE>

# Dividimos el texto en frases
frases = <INSERT CODE HERE>

# Tokenizamos todas las frases
frases_tokenizadas = <INSERT CODE HERE>

# Mostramos los tokens de las frases indicadas
print('Estos son los tokens de las frases número 3, 4 y 7:')
print(frases_tokenizadas[2])
print(frases_tokenizadas[3])
print(frases_tokenizadas[6])

# retenemos sólo los tokens de más de un caracter de largo
frases_tokenizadas_filtradas = <INSERT CODE HERE>

# Mostramos los tokens de las frases indicadas, tras el filtrado
print('\nEstos son los tokens de las frases número 3, 4 y 7 tras el filtrado:')
print(frases_tokenizadas_filtradas[2])
print(frases_tokenizadas_filtradas[3])
print(frases_tokenizadas_filtradas[6])


### Solución:

In [ ]:
# Procesamos el texto número 8
raw_text = movie_reviews.raw(doc_ids[7])

# Dividimos el texto en frases
frases = sent_tokenize(raw_text)

# Tokenizamos todas las frases
frases_tokenizadas = [word_tokenize(frase) for frase in frases]

# Mostramos los tokens de las frases indicadas
print('Estos son los tokens de las frases número 3, 4 y 7:')
print(frases_tokenizadas[2])
print(frases_tokenizadas[3])
print(frases_tokenizadas[6])

# retenemos sólo los tokens de más de un caracter de largo
frases_tokenizadas_filtradas = [[token for token in frase if len(token)>1] for frase in frases_tokenizadas]

# Mostramos los tokens de las frases indicadas, tras el filtrado
print('\nEstos son los tokens de las frases número 3, 4 y 7 tras el filtrado:')
print(frases_tokenizadas_filtradas[2])
print(frases_tokenizadas_filtradas[3])
print(frases_tokenizadas_filtradas[6])


Observamos que algunos de los tokens eliminados son "`a`", "`.`", "`,`", `?`.

## 4.2 - Homogeneización

En un texto puede haber palabras con algunas letras en mayúsculas y otras en minúsculas, unas veces aparecen en singular y otras en plural, o el mismo verbo aparece en diferentes tiempos verbales. Para analizar semánticamente el texto, nos puede interesar  **homogeneizar** las palabras que formalmente son diferentes pero tienen el mismo significado. **NOTA: Este tipo de homogeneizaciones implican una pérdida de información (plurales, tiempos verbales, etc.), por lo que no siempre tienen por qué ser beneficiosas.** El interés de aplicar este procesado se debe al problema de la **maldición de la dimensionalidad** (ing. [**Curse of dimensionality**](https://www.datacamp.com/blog/curse-of-dimensionality-machine-learning)).


El proceso habitual de homogeneización comprende los siguientes pasos:

1. **Eliminación de las mayúsculas**: de este modo los caracteres alfabéticos en mayúsculas se transformarán en sus correspondientes caracteres en minúsculas (`Hola` y `hola` pasan a ser equivalentes.)

2. **Eliminación de los caracteres no alfanuméricos** : se eliminarán los *caracteres no alfanuméricos*, por ejemplo, los *signos de puntuación*. También puede interesar eliminar los *números*, que pueden aparecer inicialmente como tokens individuales, generando mucho ruido.

3. **Stemming/Lematización**: consisten reducir las palabras **ignorando la información gramatical** (eliminamos marcas de plurales, género, conjugaciones verbales, ...), a fin de obtener una representación más sencilla. (Veremos algo más de detalle  continuación).

4. Eliminar **stop words** (palabras que en general no son muy informativas)


### Stemming y Lematización

En el lenguaje común, las palabras pueden tomar **diferentes formas indicando género, cantidad, tiempo** (en el caso de los verbos), formas concretas para nombres/adjetivos o adverbios. Para muchas aplicaciones, es útil **normalizar estas formas en alguna palabra canónica** que facilite su tratamiento. Hay dos maneras de realizar este proceso:

1. El proceso de **stemming** reduce las palabras a su base o raíz. Si bien es un proceso muy eficiente, se suelen obtener **tokens que no están en el diccionario**:

      `"compré" ->  "compr"`
      
      `"comprando" -> "compr"`
      
      `"frutícola" -> "frut"`
      
      `"frutero" -> "frut"`

  Para poder hacer esta transformación necesitamos librerías específicas que tienen almacenadas para el vocabulario de cada idioma las raíces de dicho vocabulario y hacen esta conversión.

2. El objetivo de la **lematización**, al igual que el stemming, es reducir las formas inflexionales a una forma base común. A diferencia del steming, la lematización no se limita a cortar las inflexiones. En su lugar, utiliza bases de conocimiento léxico para obtener las **formas básicas correctas de las palabras o lemas**. Es un proceso mucho más complejo y muy dependiente del lenguaje en cuestión.

    `"compré" ->  "comprar"`
    
    `"comprando" -> "comprar"`
    
    `"frutícola" -> "fruta"`
    
    `"frutero" -> "fruta"`

Una de las principales ventajas de la lematización es que **el resultado sigue siendo una palabra**
, lo que es más aconsejable para la presentación de los resultados del procesado de textos.

**Eliminar palabras irrelevantes**

El último paso del preprocesado consiste en eliminar las palabras irrelevantes o **"stop words"** de los documentos. Las **"stop words"** son las palabras más comunes en un idioma como "el", "a", "para", "por", "y", que se usan como enlaces o conexiones entre otras palabras. Estas palabras **no tienen un significado importante** y normalmente se eliminan de los textos. Para aplicar este proceso, se utilizan librerías específicas que contienen este listado de palabras por cada idioma.

Además, este paso suele utilizarse para eliminar las marcas de puntuación y otros símbolos no gramaticalmente relevantes (`',', '.', '?', '-', '+'`, etc.), para lo que también tenemos que cargar otro módulo con los signos de puntuación.


## 4.3 - Normalizado de texto con spaCy

[**spaCy**](https://spacy.io/) es una **librería gratuita de código abierto** para el procesamiento avanzado del lenguaje natural en Python. spaCy está **diseñado específicamente para uso en producción**. A diferencia de NLTK, spaCy sigue una **orientación a objetos**. Por ejemplo, cuando tokenizamos un texto, cada token es un objeto con atributos y propiedades específicas.

Spacy da **soporte a más de 64 idiomas**, incluyendo modelos estadísticos ya entrenados para [17 de ellos](https://spacy.io/usage/models) (incluyendo word embeddings y modelos basados en [transformers](https://spacy.io/usage/v3), la última revolución en NLP).


Puesto que en esta sesión sólo vamos a cubrir algunos aspectos básicos de spaCy, en los siguientes recursos podéis encontrar material adicional:

- [spaCy 101 course](https://spacy.io/usage/spacy-101)
- [Advanced Tutorial](https://course.spacy.io/en/)





El procesado de texto con spaCy es relativamente sencillo, consiste en cargar un modelo pre-entrenado para un determinado idioma, y utilizarlo para procesar los textos. **SpaCy ejecutará una serie de procesos predeterminados (pipeline) sobre el mismo y devolverá un objeto tipo `doc`**.

<figure>
<center>
<figcaption>From Spacy documentation</figcaption></center>
<center>
<img src='http://www.tsc.uc3m.es/~navia/figures/spacy_architecture.svg' width=500 />

</figure>

La arquitectura básica en spaCy contiene los siguientes elementos principales:

   - **`Language`**: **se determina al cargar el modelo** y el pipeline de procesos asociados. **Trasforma el texto en objectos spaCy**.

   - **`Vocab`**: **Diccionario asociado al modelo**. Se puede **modificar** para obtener resultados más compactos.  

   - **`Tokenizer, Lexeme`**: Son elementos de uso interno que no se suelen modificar.

   - **`Doc`**: **resultado** del proceso, es una **secuencia iterable de tokens**.

En el siguiente código, descargamos e importamos uno de los [modelos estadísticos pre-entrenados para idioma inglés](https://spacy.io/models/en)...

In [ ]:
# Descargamos un modelo para inglés (sm=small, md=medium, lg=large)
!python -m spacy download en_core_web_sm
#!python -m spacy download en_core_web_sm
#!python -m spacy download en_core_web_sm

# 2026:
# Collecting en-core-web-sm==3.8.0
# Collecting en-core-web-md==3.8.0
# Collecting en-core-web-lg==3.8.0

In [ ]:
# Cargamos el modelo
nlp = spacy.load("en_core_web_sm")

Podemos ver los componentes activos en el "pipeline" con ``nlp.component_names``:

In [ ]:
print(nlp.component_names)
#['tok2vec', 'tagger', 'parser', 'senter', 'attribute_ruler', 'lemmatizer', 'ner']


El **acceso a las palabras del vocabulario** se hace a través del atributo `.vocab.strings`

In [ ]:
lista_vocab = list(nlp.vocab.strings)
print(type(nlp.vocab.strings))
print(type(lista_vocab))

print("\nEl tamaño del diccionario es de {} 'palabras' de todo tipo, algunos ejemplos:".format(len(lista_vocab)))

#Las 20 primeras palabras y las 20 en posición 1000 (muchas son alfanuméricas)

print(lista_vocab[:20])
print(lista_vocab[1000:1020])
print(lista_vocab[10000:10020])

El **tamaño del vocabulario puede depender del modelo cargado**: "sm" tiene 84.780 tokens en el vocabulario, mientras que "md" y "lg" tienen 709.118 tokens.  

In [ ]:
# Retenemos únicamente los tokens alfabéticos
# NOTA: .isalpha() es un método del tipo string

alpha_words = [word for word in lista_vocab if word.isalpha()]
no_alpha_words = [word for word in lista_vocab if not word.isalpha()]

print(f'Hay un total de {len(no_alpha_words)} palabras con caracteres no alfabéticos. Algunas de ellas son:')
print(no_alpha_words[1000:1010])

print(f'\nHay un total de {len(alpha_words)} palabras sólo con caracteres alfabéticos. Algunos ejemplos:')
print(alpha_words[1000:1010])

### Objetos de texto en SpaCy

A continuación, vamos a utilizar el pipeline que hemos cargado para analizar un texto.

**NOTA: el formato de entrada a Spacy siempre es un texto plano.**

In [ ]:
sentence = 'Japan’s economy, struggling to emerge from the pandemic, is sagging under rising food and energy costs and a weak yen'

print("Esta es la frase de partida, tipo `string`:\n")
print(sentence)
print(type(sentence))

doc_tokenized = nlp(sentence)

print("\nAunque al hacer un `print` se nos muestra el texto, vemos que no es un string habitual...\n")
print(doc_tokenized)
print(type(doc_tokenized))

`doc_tokenized` es un objeto indexado e iterable, compuesto por objetos tipo [`token`](https://spacy.io/api/token).

In [ ]:
print(doc_tokenized[0])
print(type(doc_tokenized[0]))

Veamos algunos de los métodos o atributos disponibles para cada token:

In [ ]:
for attribute in dir(doc_tokenized[0]):
  if "__" not in attribute:
    print(attribute)

Los atributos del tipo "`is_`" nos serán especialmente útiles, por ejemplo **`is_digit`** para reconocer números, **`is_alpha`** para reconocer tokens alfabéticos, **`is_stop`** para detectar las stop words, etc. En el siguiente bucle imprimimos algunas de las propiedades de dichos tokens determinadas por el pipeline que hemos cargado, incluyendo el **POS tag** ("Part Of Speech"):

In [ ]:
for token in doc_tokenized:
    print(token.text, token.lemma_, token.tag_, token.is_alpha, token.is_stop, token.is_punct)
    print(30 * '-')

Los resultados se visualizan mejor en forma de **tabla**, con la ayuda de **Pandas**:

In [ ]:
spacy_pos_tagged = [(token.text, token.lemma_, token.tag_, token.is_alpha, token.is_stop,token.is_punct) for token in doc_tokenized]

pd.DataFrame(spacy_pos_tagged, columns=['Word','Word_Lemma','POS_tag','Is_alpha','Is_stopword','Is_punct'])

Con `spacy.explain()` podemos obtener una descripción de los distintos tags ...

In [ ]:
spacy.explain("VBP")

In [ ]:
spacy.explain("JJ")

### Ejercicio:

Mostrar el significado de cada uno de los POS tags presentes en la frase anterior.

In [ ]:
POS_taggs = <INSERT CODE HERE>
POS_taggs.sort()

for tag in POS_taggs:
  print(<INSERT CODE HERE>)

### Solución:

In [ ]:
POS_taggs = list(set([token.tag_ for token in doc_tokenized]))
POS_taggs.sort()

for tag in POS_taggs:
  print(tag, "->", spacy.explain(tag))

En lo referente a **stop words** (palabras vacías), spaCy incluye una amplia lista (326 elementos en idioma inglés) que podemos personalizar para nuestra propia aplicación si es necesario.

In [ ]:
spacy_stopwords = nlp.Defaults.stop_words

#Printing the total number of stop words:
print('Number of stop words: %d' % len(spacy_stopwords))

#Printing first ten stop words:
print('First ten stop words: %s' % list(spacy_stopwords)[:20])

In [ ]:
# Añadir una palabra al conjunto de stop words

nlp.Defaults.stop_words.add("my_new_stopword")

print('Number of stop words: %d' % len(nlp.Defaults.stop_words))

# Quitar una palabra del conjunto de stop words

nlp.Defaults.stop_words.remove("my_new_stopword")

print('Number of stop words: %d' % len(nlp.Defaults.stop_words))



## 4.4 - Spacy para idioma español

Tal y como hemos comentado, **Spacy proporciona modelos pre-entrenados para trabajar con idioma español**. Todos han sido entrenados en la base de datos anotada [Ancora](http://clic.ub.edu/corpus/). Este corpus contiene 500.000 textos  periodísticos publicados en medios españoles.

Es recomendable mirar la siguiente [documentación](https://spacy.io/models/es).


In [ ]:
!python -m spacy download es_core_news_sm

# 2026:
# Collecting es-core-news-sm==3.8.0
# Collecting es-core-news-md==3.8.0
# Collecting es-core-news-lg==3.8.0

In [ ]:
nlp = spacy.load("es_core_news_sm")

lista_vocab = list(nlp.vocab.strings)

print("El tamaño del diccionario es de {} palabras. Las primeras 10 tras la posición 20000 son \n".format(len(lista_vocab)))

print(lista_vocab[20000:20010])

#Printing the total number of stop words:
print('\nHay un total de {0} stop words en el modelo'.format(len(spacy.lang.es.stop_words.STOP_WORDS)))
print(list(spacy.lang.es.stop_words.STOP_WORDS)[0:20])



In [ ]:
texto = "Spacy proporciona modelos pre-entrenados para trabajar con idioma español. Todos han sido entrenados en la base de datos anotada Ancora. Este corpus contiene 500.000 textos periodísticos publicados en medios españoles."

In [ ]:
texto_tokenizado = nlp(texto)

In [ ]:
spacy_pos_tagged = [(token.text, token.lemma_, token.tag_) for token in texto_tokenizado if token.is_alpha]

pd.DataFrame(spacy_pos_tagged, columns=['Word','Word_Lemma','POS_tag'])

## 4.5 - Uso de SpaCy para el preprocesamiento de texto

Como hemos visto, usando SpaCy, podemos acceder directamente a los lemas de cada token junto con algunos atributos para indicar si el token es una palabra vacía (.is_stop), alfanumérico (.is_alpha), un signo de puntuación (.is_punct) o un dígito (. es_dígito) . Podemos usar directamente esta información para preprocesar nuestro corpus.

La siguiente función implementa una canalización de normalización para un documento determinado:

In [ ]:
print('Volvemos a cargar el modelo para inglés a fin de procesar el corpus de películas:')
nlp = spacy.load("en_core_web_sm")

El procesado de textos usando el **pipeline completo de Spacy puede ser muy costoso**. Por ello, suele ser conveniente **habilitar o deshabilitar** partes del proceso que no se vayan a utilizar, tal y como se describe [aquí](https://spacy.io/models/#design-modify). Es decir, podemos añadir o eliminar elementos del pipeline usando:

- ``nlp.add_pipe(p)``: añade proceso
- ``nlp.disable_pipe(p)``: elimina proceso

donde ``p`` puede ser uno de los siguientes **procesos spacy**:

- "tok2vec"
- "tagger"
- "parser"
- "senter"
- "attribute_ruler"
- "lemmatizer"
- "ner"

Por tanto, primero decidimos cuál va a ser el preprocesado que vamos a realizar:

In [ ]:
def normalize_Spacy(text):
    text2 = nlp(text)
    # Ejercicio: interpretar el sentido de esta list comprehension:
    normalized_text = [w.lemma_.lower() for w in text2 if not w.is_stop
                  and not w.is_punct and (w.is_alpha or w.is_digit)]
    return normalized_text

A continuación desabilitamos aquéllos procesos que no vamos a utilizar, para acelerar un poco la ejecución:

In [ ]:
#nlp.disable_pipes('tok2vec', 'tagger', 'parser', 'senter', 'attribute_ruler', 'lemmatizer', 'ner')
nlp.disable_pipes('ner')
nlp.disable_pipes('tok2vec')

Y ahora normalizamos todo el corpus (**esta operación puede llevar más tiempo si no se eliminan los procesos innecesarios...**). Si durante la normalización observamos algún mensaje de aviso, puede que hayamos eliminado algún componente necesario y toca restituirlo.

In [ ]:
corpus_prec = []

for fileid in movie_reviews.fileids():
    text = movie_reviews.raw(fileid)
    text_preproc = normalize_Spacy(text)
    corpus_prec.append(text_preproc)

In [ ]:
print("corpus_prec es ahora una lista de listas, conteniendo {} documentos tokenizados y homogeneizados.".format(len(corpus_prec)))

Estas son las 20 primeras palabras normalizadas en el primer documento del corpus:

In [ ]:
corpus_prec[0][0:20]

## 4.6 - Contando las palabras más frecuentes en nuestro corpus

Utilizando la clase `Counter` dentro de la librería `collections` podemos encontrar facilmente las palabras más utilizadas en nuestro corpus ya preprocesado:

In [ ]:
from collections import Counter

all_normalized_words = [w for d in corpus_prec for w in d]
words_freq = Counter(all_normalized_words)

In [ ]:
common_words = words_freq.most_common(20)

In [ ]:
for pair in common_words[0:20]:
  print(pair)

### Ejercicio

Obtenga la lista con los 20 adjetivos más frecuentes en el corpus

In [ ]:
# We select the alpha words in the vocabulary, in lower case and eliminating possible duplicates
vocab_list_alpha_words = <INSERT CODE HERE>
print("Hay un total de {} palabras alfanuméricas en el corpus.".format(len(vocab_list_alpha_words)))
print(vocab_list_alpha_words[0:100])

# We make a list with all the words in the movies corpus, in lower case
all_words = <INSERT CODE HERE>
print("\nHay un total de {} palabras en el corpus.".format(len(all_words)))

In [ ]:
# We select the words in the vocabulary that are adjectives
adjectives_list = <INSERT CODE HERE>
print("Hay un total de {} adjetivos en el corpus.".format(len(adjectives_list)))
print(adjectives_list[0:100])
print(adjectives_list[:100])

In [ ]:
# We filter the words in the corpus, retaining only the adjectives:
adjectives_in_corpus = <INSERT CODE HERE>
print("Hay un total de {} adjetivos en el corpus, algunos de ellos son:".format(len(adjectives_in_corpus)))
print(adjectives_in_corpus[:100])

In [ ]:
# Now we count the adjectives in the corpus and print the most common ones
print("Esta es la lista de los 20 adjetivos más comunes, así como su conteo en el corpus:")
adjectives_frequency = <INSERT CODE HERE>
most_common_adjectives = <INSERT CODE HERE>
for word, freq in most_common_adjectives[:20]:
    print(word, freq)


### Solución

In [ ]:
# We make a list with all the words in the movies corpus, in lower case
all_words = [w.lower() for d in corpus_prec for w in d]
print("\nHay un total de {} palabras en el corpus.".format(len(all_words)))

In [ ]:
# We select the alpha words in the vocabulary, in lower case and eliminating possible duplicates
vocab_list_alpha_words = list(set([word.lower() for word in nlp.vocab.strings if word.isalpha()]))
print("\nHay un total de {} palabras alfanuméricas en el vocabulario.".format(len(vocab_list_alpha_words)))
print(vocab_list_alpha_words[0:100])

In [ ]:
# Para identificar los adjetivos, volvemos a cargar el pipeline nlp completo
nlp = spacy.load("en_core_web_sm")

# We select the words in the vocabulary that are adjectives
adjectives_list = [word for word in vocab_list_alpha_words if nlp(word)[0].tag_ == "JJ"]
print("\nHay un total de {} adjetivos en el vocabulario.".format(len(adjectives_list)))
print(adjectives_list[:100])

In [ ]:
# We filter the words in the corpus, retaining only the adjectives:
adjectives_in_corpus = [word for word in all_words if word in adjectives_list]
print("Hay un total de {} adjetivos en el corpus, algunos de ellos son:".format(len(adjectives_in_corpus)))
print(adjectives_in_corpus[:100])

In [ ]:
# Now we count the adjectives in the corpus and print the most common ones
print("\nEsta es la lista de los 20 adjetivos más comunes, así como su conteo en el corpus:")
adjectives_frequency = Counter(adjectives_in_corpus)
most_common_adjectives = adjectives_frequency.most_common(20)
for word, freq in most_common_adjectives[:20]:
    print(word, freq)

### Ejercicio de ampliación

- Localice todos los **pares de palabras contiguas** de tipo **verbo + sustantivo**, y ordénelos según su frecuencia de aparición en el corpus.
- Repita el proceso **filtrando** previamente los tipos de palabras que puedan resultar superfluos para esta tarea, por ejemplo: "go to the cinema" -> "go cinema".
